### Analysis suite

Version 1
- word count
- n-grams
- tf-idf (log)
- collocation explorer

Version 2
- topic clusters
- semantic similarity
- embeddings?

In [ ]:
%pip install spacy

In [2]:
import re
import nltk # used for tokenization, which is important to clean the text and to prepare it for the language model
import pandas as pd # used to work with dataframes, which is important to store the data in a structured way and to manipulate it easily
import matplotlib.pyplot as plt # used to visualize the data, which is important to understand it and to communicate the results
import gensim # used for topic modeling, which is important to extract the main topics from the laws and to understand their content
import spacy # used for named entity recognition, which is important to extract the entities from the laws and to understand their content
from nltk.corpus import stopwords # used to remove stopwords

In [1]:
LOCATION_SPEECHES = ("../../data/processed/turns.csv") # change name
LOCATION_DB = ("../../data/chilean_polispeech.db/turns.db")

In [3]:
clean_speech = pd.read_csv(LOCATION_SPEECHES)

In [4]:
clean_speech

,law_number,section_key,turn_order,speaker_header_raw,speaker_article,speaker_label,speaker_body_raw,speech_raw,role_raw,turn_type,speech_clean
0,21.472,1-5,1,"El señor SOTO, don Raúl (Presidente).-",El,señor,"SOTO, don Raúl (Presidente)","Iniciando el Orden del Día, corresponde tratar...",Presidente,floor_control,NaN
1,21.472,1-5,2,"El señor SOTO, don Raúl (Presidente).-",El,señor,"SOTO, don Raúl (Presidente)","Doy la bienvenida al ministro de Energía, seño...",Presidente,floor_control,NaN
2,21.472,1-5,3,"La señorita AHUMADA, doña Yovana (de pie).-",La,señorita,"AHUMADA, doña Yovana (de pie)","Señor Presidente, en mi calidad de diputada in...",de pie,law_introduction,NaN
3,21.472,1-5,4,"El señor SOTO, don Raúl (Presidente).-",El,señor,"SOTO, don Raúl (Presidente)",Tiene la palabra el diputado informante de la ...,Presidente,floor_control,NaN
4,21.472,1-5,5,El señor NARANJO (de pie).-,El,señor,NARANJO (de pie),"Señor Presidente, honorable Cámara, en nombre ...",de pie,floor_control,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1523,21.800,2-5,82,El señor AEDO (Vicepresidente).-,El,señor,AEDO (Vicepresidente),Corresponde votar en particular el artículo 21...,Vicepresidente,floor_control,NaN
1524,21.800,2-5,83,El señor AEDO (Vicepresidente).-,El,señor,AEDO (Vicepresidente),Aprobado.\n-Votaron por la afirmativa:\nAceved...,Vicepresidente,floor_control,NaN
1525,21.800,2-5,84,El señor AEDO (Vicepresidente).-,El,señor,AEDO (Vicepresidente),Corresponde votar en particular el artículo te...,Vicepresidente,floor_control,NaN
1526,21.800,2-5,85,El señor AEDO (Vicepresidente).-,El,señor,AEDO (Vicepresidente),Aprobado.\n-Votaron por la afirmativa:\nAceved...,Vicepresidente,floor_control,NaN


Creation of enriched dataset

speech + deputies data

In [ ]:
#to the data set

Analysis suite

# old analysis

## Speech analysis


Analyze the extracted speeches to gain insights into topics discussed, speech frequency, length, and any relevant patterns.
Link these insights to the deputies’ metadata (party affiliation, region, etc.) to uncover trends or biases in the discussions.

### LDA

In [ ]:
# Download Spanish stopwords
nltk.download('stopwords', quiet=True)

# Read the CSV file
df = pd.read_csv("extracted_speeches.csv")

# Get Spanish stop words and add more names/common words
additional_stop_words = {'jorge', 'diputado', 'juan', 'palabra', 'maría', 'martínez', 'jaime', 'miguel', 'castro', 'pérez', 'señor', 'señora', 'gonzález', 'andrés', 'muñoz', 'castillo', 'luis', 'araya', 'sáez', 'moreno', 'romero', 'votación'}
spanish_stop_words = set(stopwords.words('spanish')).union(additional_stop_words)

# Text Preprocessing Function
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    words = text.split()
    words = [word for word in words if word not in spanish_stop_words]
    return ' '.join(words)

# Apply preprocessing to speeches
df['Processed_Speech'] = df['Speech'].apply(preprocess_text)

# Combine all speeches into a single document for each speaker
df_combined = df.groupby('Speaker')['Processed_Speech'].apply(' '.join).reset_index()

# Prepare the data for Gensim
texts = df_combined['Processed_Speech'].apply(lambda x: x.split()).tolist()
id2word = corpora.Dictionary(texts)
corpus = [id2word.doc2bow(text) for text in texts]

# Compute coherence values for different numbers of topics
def compute_coherence_values(dictionary, corpus, texts, limit, start=2, step=1):
    coherence_values = []
    model_list = []
    for num_topics in range(start, limit + 1, step):
        model = gensim.models.LdaMulticore(corpus=corpus, num_topics=num_topics, id2word=dictionary, passes=15, iterations=100)
        model_list.append(model)
        coherencemodel = CoherenceModel(model=model, texts=texts, dictionary=dictionary, coherence='c_v')
        coherence_values.append(coherencemodel.get_coherence())
    return model_list, coherence_values

# Compute coherence values
start, limit, step = 2, 10, 1
model_list, coherence_values = compute_coherence_values(dictionary=id2word, corpus=corpus, texts=texts, start=start, limit=limit, step=step)

# Plot coherence scores
x = range(start, limit + 1, step)
plt.plot(x, coherence_values)
plt.xlabel("Num Topics")
plt.ylabel("Coherence Score")
plt.title("Coherence Scores for Different Numbers of Topics")
plt.show()

# Print coherence scores
for m, cv in zip(x, coherence_values):
    print(f"Num Topics = {m}, Coherence Score = {cv}")

# Find the optimal number of topics
optimal_num_topics = x[coherence_values.index(max(coherence_values))]
print(f"Optimal number of topics: {optimal_num_topics}")

# Train the LDA model with the optimal number of topics
lda_optimal = gensim.models.LdaMulticore(corpus=corpus, num_topics=optimal_num_topics, id2word=id2word, passes=15, iterations=100)

# Function to print top words for each topic
def print_gensim_topics(model, n_top_words):
    topics = model.show_topics(num_topics=optimal_num_topics, num_words=n_top_words, formatted=False)
    for idx, topic in topics:
        print(f"Topic {idx + 1}: {', '.join([word for word, _ in topic])}")

# Print the top 5 words for each topic
print("\nTop 5 words for each topic:")
print_gensim_topics(lda_optimal, 5)

def plot_gensim_topics(model, n_top_words):
    topics = model.show_topics(num_topics=optimal_num_topics, num_words=n_top_words, formatted=False)

    for idx, topic in topics:
        # Extract the top words and their weights
        words, weights = zip(*topic)

        # Plot the words and weights for the topic
        plt.figure(figsize=(6, 2))  # Reduced figsize for smaller plots
        plt.barh(words, weights, color='skyblue')
        plt.gca().invert_yaxis()  # Invert y-axis for better readability
        plt.title(f"Topic {idx + 1}: Top {n_top_words} Words")
        plt.xlabel("Word Weight")
        plt.ylabel("Words")
        plt.tight_layout()
        plt.show()

# Plot the top 5 words for each topic with smaller plots
plot_gensim_topics(lda_optimal, 5)


In [ ]:
# Download Spanish stopwords
nltk.download('stopwords')

# Read the CSV file
df = pd.read_csv("matched_speeches.csv")

# Get Spanish stop words and add more names/common words
additional_stop_words = {'jorge', 'diputado', 'juan', 'palabra', 'maría', 'martínez', 'jaime', 'miguel', 'castro', 'pérez', 'señor', 'señora', 'gonzález', 'andrés', 'muñoz', 'castillo', 'luis', 'araya', 'sáez', 'moreno', 'romero', 'votación'}
spanish_stop_words = set(stopwords.words('spanish')).union(additional_stop_words)

# Text Preprocessing Function
def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Remove stopwords
    stop_words = set(stopwords.words('spanish'))
    words = text.split()
    words = [word for word in words if word not in spanish_stop_words]
    return ' '.join(words)

# Apply preprocessing to speeches
df['Processed_Speech'] = df['Speech'].apply(preprocess_text)

# Basic Analysis
print(f"Total number of speeches: {len(df)}")
# Instead of 'Speaker', use 'Full Name' to get unique speakers
print(f"Number of unique speakers: {df['Full Name'].nunique()}")

# Top 10 speakers by number of speeches
top_speakers = df['Full Name'].value_counts().head(10)
print("\nTop 10 speakers by number of speeches:")
print(top_speakers)

# Word frequency across all speeches (remains the same)
all_words = ' '.join(df['Processed_Speech']).split()
word_freq = Counter(all_words)
print("\nTop 20 most common words:")
print(word_freq.most_common(20))

# Visualize top speakers
plt.figure(figsize=(12, 6))
sns.barplot(x=top_speakers.values, y=top_speakers.index)
plt.title('Top 10 Speakers by Number of Speeches')
plt.xlabel('Number of Speeches')
plt.ylabel('Full Name')  # Changed ylabel
plt.tight_layout()
plt.savefig('top_speakers.png')
plt.show()
plt.close()

# Visualize top words (remains the same)
top_words = dict(word_freq.most_common(20))
plt.figure(figsize=(12, 6))
sns.barplot(x=list(top_words.values()), y=list(top_words.keys()))
plt.title('Top 20 Most Common Words')
plt.xlabel('Frequency')
plt.ylabel('Word')
plt.tight_layout()
plt.savefig('top_words.png')
plt.show()
plt.close()

print("\nAnalysis complete. Check 'top_speakers.png' and 'top_words.png' for visualizations.")

### Name Entity Recognition and Sentiment Analysis

In [ ]:
# Load Spanish language model
nlp = spacy.load("es_core_news_sm")
nlp.add_pipe("spacytextblob")

def analyze_speech(text):
    doc = nlp(text)

    # Sentiment Analysis
    sentiment = doc._.blob.polarity

    # Named Entity Recognition
    entities = {
        'PER': [],
        'ORG': [],
        'LOC': []
    }
    for ent in doc.ents:
        if ent.label_ in entities:
            entities[ent.label_].append(ent.text)

    return sentiment, entities

def process_speeches(file_path):
    # Read the CSV file
    df = pd.read_csv(file_path)

    # Apply analysis to each speech
    results = df['Speech'].apply(analyze_speech)

    # Split results into separate columns
    df['Sentiment'] = results.apply(lambda x: x[0])
    df['Entities'] = results.apply(lambda x: x[1])

    return df

def visualize_sentiment(df):
    plt.figure(figsize=(10, 6))
    sns.histplot(df['Sentiment'], kde=True)
    plt.title('Distribution of Sentiment Scores')
    plt.xlabel('Sentiment Score')
    plt.ylabel('Frequency')
    plt.savefig('sentiment_distribution.png')
    plt.show()
    plt.close()

def analyze_entities(df):
    all_entities = {'PER': [], 'ORG': [], 'LOC': []}
    for entities in df['Entities']:
        for key in all_entities:
            all_entities[key].extend(entities[key])

    for entity_type in all_entities:
        entity_counts = pd.Series(all_entities[entity_type]).value_counts().head(10)
        plt.figure(figsize=(10, 6))
        sns.barplot(x=entity_counts.index, y=entity_counts.values)
        plt.title(f'Top 10 {entity_type} Entities')
        plt.xlabel('Entity')
        plt.ylabel('Frequency')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(f'top_{entity_type}_entities.png')
        plt.show()
        plt.close()

def main():
    file_path = '/content/extracted_speeches.csv'
    df = process_speeches(file_path)

    # Save processed data
    df.to_csv('analyzed_speeches.csv', index=False)

    # Visualize results
    visualize_sentiment(df)
    analyze_entities(df)

    print("\n=== Sentiment Summary ===")
    print(df['Sentiment'].describe())

    # Entity counts
    print("\n=== Top Entities ===")
    from collections import Counter
    all_ents = Counter()
    for dic in df['Entities']:
        for label, lst in dic.items():
            all_ents[label] += len(lst)
    for label in ['PER','ORG','LOC']:
        print(f"{label}: {all_ents[label]} mentions")

    print("Analysis complete. Results saved to 'analyzed_speeches.csv' and visualization images.")

if __name__ == "__main__":
    main()



---



Sentiments ordered by coalitions

In [ ]:
# Load the Spanish language model and add the sentiment analysis component
nlp = spacy.load("es_core_news_sm")
nlp.add_pipe("spacytextblob")

# Read the matched speeches data
df = pd.read_csv("matched_speeches.csv")

# Function to get sentiment
def get_sentiment(text):
    doc = nlp(text)
    return doc._.blob.polarity

# Add sentiment to each row if not already present
if 'Sentiment' not in df.columns:
    df['Sentiment'] = df['Speech'].apply(get_sentiment)

# Calculate average sentiment for each political category
coalition_sentiment = df.groupby('Political Category')['Sentiment'].mean().reset_index()
coalition_sentiment.columns = ['Political Category', 'Avg_Sentiment']

# Print the results
print(coalition_sentiment)

# Save the updated DataFrame with sentiment scores
df.to_csv("matched_speeches_with_sentiment.csv", index=False)

# Filter out "Unknown" category
coalition_sentiment = coalition_sentiment[coalition_sentiment['Political Category'] != 'Unknown']

# Define the desired order of political categories
political_order = ['Alianza de Gobierno', 'Center', 'Center-Right', 'Right Wing']

# Create the bar plot
plt.figure(figsize=(10, 6))
sns.barplot(
    x='Political Category',
    y='Avg_Sentiment',
    data=coalition_sentiment,
    order=political_order  # Enforce the desired order
)
plt.title('Average Sentiment by Political Category (Left to Right)')
plt.xlabel('Political Category')
plt.ylabel('Average Sentiment Score')
plt.xticks(rotation=45, ha='right')  # Rotate x-axis labels for readability
plt.tight_layout()
plt.show()

### Term Frequency-Inverse Document Frequency

In [ ]:
additional_stop_words = set([
    'jorge', 'diputado', 'juan', 'palabra', 'maría', 'martínez', 'jaime', 'miguel', 'castro', 'pérez',
    'señor', 'señora', 'gonzález', 'andrés', 'muñoz', 'castillo', 'luis', 'araya', 'sáez', 'moreno',
    'romero', 'votación', 'camila', 'soto', 'salazar', 'salinas', 'daniel', 'ramírez', 'cristián',
    'antonio', 'leiva', 'sánchez', 'raúl', 'undurraga', 'guzmán', 'francisco', 'felipe', 'venegas',
    'marta', 'ángel', 'riquelme', 'alberto', 'bravo', 'mellado', 'emilia', 'ana', 'tomás', 'eduardo',
    'cristóbal', 'videla', 'gonzalo', 'maldonado', 'catalina', 'carlos', 'espinoza', 'palma', 'olea',
    'durán', 'valderrama', 'flores', 'vergara', 'félix', 'héctor', 'rojas', 'diputada', 'diputado',
    'vicepresidenta', 'vicepresidente', 'presidente', "quiero", "decir"
])
spanish_stop_words = set(stopwords.words('spanish')).union(additional_stop_words)

# Calculate TF-IDF with stopwords
tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words=list(spanish_stop_words))
tfidf_matrix = tfidf_vectorizer.fit_transform(matched_speeches['Speech'])

# Get feature names (words)
feature_names = tfidf_vectorizer.get_feature_names_out()

def get_top_tfidf_words(tfidf_matrix, feature_names, top_n=10):
    sums = tfidf_matrix.sum(axis=0).A1
    term_scores = [(term, score) for term, score in zip(feature_names, sums)]
    term_scores = sorted(term_scores, key=lambda x: x[1], reverse=True)
    return term_scores[:top_n]

# Get top TF-IDF words overall
top_words = get_top_tfidf_words(tfidf_matrix, feature_names)
print("Top TF-IDF words overall:")
for word, score in top_words:
    print(f"{word}: {score}")

# Get top TF-IDF words by political category
for category in matched_speeches['Political Category'].unique():
    category_speeches = matched_speeches[matched_speeches['Political Category'] == category]
    category_tfidf = tfidf_vectorizer.transform(category_speeches['Speech'])
    top_words = get_top_tfidf_words(category_tfidf, feature_names)
    print(f"\nTop TF-IDF words for {category}:")
    for word, score in top_words:
        print(f"{word}: {score}")

In [ ]:
# Define the coalitions
gov_coalition = ['Alianza de Gobierno']
right_coalition = ['Center-Right', 'Right Wing']

# Create a new column grouping categories into two main coalitions
matched_speeches['Coalition'] = matched_speeches['Political Category'].apply(
    lambda x: 'Government Coalition' if x in gov_coalition else 'Right Coalition' if x in right_coalition else 'Other'
)

# Filter speeches for Government Coalition
gov_speeches = matched_speeches[matched_speeches['Coalition'] == 'Government Coalition']

# Filter speeches for Right Coalition
right_speeches = matched_speeches[matched_speeches['Coalition'] == 'Right Coalition']

# Calculate TF-IDF for Government Coalition
gov_tfidf = tfidf_vectorizer.fit_transform(gov_speeches['Speech'])
gov_feature_names = tfidf_vectorizer.get_feature_names_out()
gov_top_words = get_top_tfidf_words(gov_tfidf, gov_feature_names, top_n=10)

# Calculate TF-IDF for Right Coalition
right_tfidf = tfidf_vectorizer.fit_transform(right_speeches['Speech'])
right_feature_names = tfidf_vectorizer.get_feature_names_out()
right_top_words = get_top_tfidf_words(right_tfidf, right_feature_names, top_n=10)

# Print top words for Government Coalition
print("\nTop TF-IDF words for Government Coalition:")
for word, score in gov_top_words:
    print(f"{word}: {score}")

# Print top words for Right Coalition
print("\nTop TF-IDF words for Right Coalition:")
for word, score in right_top_words:
    print(f"{word}: {score}")

import matplotlib.pyplot as plt
import pandas as pd

# Create a DataFrame for comparison
gov_df = pd.DataFrame(gov_top_words, columns=['Word', 'Score'])
right_df = pd.DataFrame(right_top_words, columns=['Word', 'Score'])

# Plot Government Coalition
plt.figure(figsize=(6, 2))
plt.barh(gov_df['Word'], gov_df['Score'], color='blue', label='Government Coalition')
plt.title('Top TF-IDF Words: Government Coalition')
plt.xlabel('TF-IDF Score')
plt.ylabel('Words')
plt.show()

# Plot Right Coalition
plt.figure(figsize=(6, 2))
plt.barh(right_df['Word'], right_df['Score'], color='red', label='Right Coalition')
plt.title('Top TF-IDF Words: Right Coalition')
plt.xlabel('TF-IDF Score')
plt.ylabel('Words')
plt.show()
